# Assignment 33 : Chat with SQL Database using Langchain

# PART 1 - Storing Data in SQLite Database

## Task 1 : Create SQLite Database

In [1]:
import sqlite3

In [2]:
connection = sqlite3.connect('company.db')

In [3]:
cursor = connection.cursor()

In [4]:
# cursor.execute('''CREATE TABLE EMPLOYEES( ID INTEGER PRIMARY KEY, NAME VARCHAR(20), DEPARTMENT VARCHAR(20), SALARY INT)''')
# cursor.execute('''CREATE TABLE SALES( SALES_ID INTEGER PRIMARY KEY, EMPLOYEE_ID INT, AMOUNT INT, SALES_DATE DATE)''')

In [5]:
cursor.execute('''SELECT name FROM sqlite_master WHERE type='table' ''')
tables = cursor.fetchall()

In [6]:
tables

[('EMPLOYEES',), ('SALES',)]

## Task 2 : Insert Sample Data

In [7]:
dummy_employees = [
    (101, 'Amit Sharma', 'Engineering', 85000),
    (102, 'Priya Singh', 'Marketing', 62000),
    (103, 'Rohit Verma', 'Sales', 55000),
    (104, 'Neha Gupta', 'HR', 75000),
    (105, 'Vikram Patel', 'Engineering', 90000),
    (106, 'Deepak Kumar', 'Finance', 82000),
    (107, 'Sneha Reddy', 'Engineering', 78000),
    (108, 'Rohan Das', 'Operations', 48000),
    (109, 'Anjali Mehra', 'Marketing', 65000),
    (110, 'Karthik Iyer', 'Finance', 95000)
]

In [8]:
cursor.executemany('''
    INSERT OR IGNORE INTO EMPLOYEES (ID, NAME, DEPARTMENT, SALARY) 
    VALUES (?, ?, ?, ?)
''', dummy_employees)

In [9]:
emp_data = cursor.execute('''SELECT * FROM EMPLOYEES''')

In [10]:
for row in emp_data:
    print(row)

(101, 'Amit Sharma', 'Engineering', 85000)
(102, 'Priya Singh', 'Marketing', 62000)
(103, 'Rohit Verma', 'Sales', 55000)
(104, 'Neha Gupta', 'HR', 75000)
(105, 'Vikram Patel', 'Engineering', 90000)
(106, 'Deepak Kumar', 'Finance', 82000)
(107, 'Sneha Reddy', 'Engineering', 78000)
(108, 'Rohan Das', 'Operations', 48000)
(109, 'Anjali Mehra', 'Marketing', 65000)
(110, 'Karthik Iyer', 'Finance', 95000)


In [11]:
dummy_sales = [
    (101, 3, 15000, '2026-01-05'),
    (102, 3, 22000, '2026-01-12'),
    (103, 8, 8500,  '2026-01-15'),
    (104, 2, 31000, '2026-01-20'),
    (105, 9, 12500, '2026-02-02'),
    (106, 3, 19000, '2026-02-10'),
    (107, 8, 14000, '2026-02-18'),
    (108, 9, 27500, '2026-02-25'),
    (109, 2, 16000, '2026-03-01'),
    (110, 8, 9200,  '2026-03-05')
]

In [12]:
cursor.executemany('''
    INSERT OR IGNORE INTO SALES (SALES_ID, EMPLOYEE_ID, AMOUNT, SALES_DATE) 
    VALUES (?, ?, ?, ?)
''', dummy_sales)

In [13]:
sales_data = cursor.execute('''SELECT * FROM SALES''')

In [14]:
for row in sales_data:
    print(row)

(101, 3, 15000, '2026-01-05')
(102, 3, 22000, '2026-01-12')
(103, 8, 8500, '2026-01-15')
(104, 2, 31000, '2026-01-20')
(105, 9, 12500, '2026-02-02')
(106, 3, 19000, '2026-02-10')
(107, 8, 14000, '2026-02-18')
(108, 9, 27500, '2026-02-25')
(109, 2, 16000, '2026-03-01')
(110, 8, 9200, '2026-03-05')


In [15]:
connection.commit()

# PART 2 - Creating Langchain Database Engine

## Task 3 : Create SQLAlchemy Engine

In [16]:
from sqlalchemy import create_engine, inspect
from pathlib import Path

In [17]:
engine = create_engine('sqlite:///company.db')

In [23]:
from sqlalchemy import text

with engine.connect() as connection:
    print('Connection Successfully extablised')
    result = connection.execute(text("""SELECT name FROM sqlite_master WHERE type='table'"""))

Connection Successfully extablised


In [24]:
tables = result.fetchall()

In [25]:
tables

[('EMPLOYEES',), ('SALES',)]

## Task 4 : Create Langchain SQLDatabase Object

In [26]:
from langchain_community.utilities.sql_database import SQLDatabase
database = SQLDatabase.from_uri('sqlite:///company.db')

/var/folders/cf/t0bj31ws20944pwkbddw0ylh0000gn/T/ipykernel_96946/106478950.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [27]:
tables = database.get_usable_table_names()

In [28]:
for table in tables:
    print(table)

EMPLOYEES
SALES


In [29]:
schema = database.get_table_info(table_names=['EMPLOYEES','SALES'])

In [30]:
print(schema)


CREATE TABLE "EMPLOYEES" (
	"ID" INTEGER, 
	"NAME" VARCHAR(20), 
	"DEPARTMENT" VARCHAR(20), 
	"SALARY" INTEGER, 
	PRIMARY KEY ("ID")
)

/*
3 rows from EMPLOYEES table:
ID	NAME	DEPARTMENT	SALARY
101	Amit Sharma	Engineering	85000
102	Priya Singh	Marketing	62000
103	Rohit Verma	Sales	55000
*/


CREATE TABLE "SALES" (
	"SALES_ID" INTEGER, 
	"EMPLOYEE_ID" INTEGER, 
	"AMOUNT" INTEGER, 
	"SALES_DATE" DATE, 
	PRIMARY KEY ("SALES_ID")
)

/*
3 rows from SALES table:
SALES_ID	EMPLOYEE_ID	AMOUNT	SALES_DATE
101	3	15000	2026-01-05
102	3	22000	2026-01-12
103	8	8500	2026-01-15
*/


In [31]:
emp_data = database._execute(text('''SELECT * FROM EMPLOYEES'''))

In [34]:
for row in emp_data:
    print(row)

{'ID': 101, 'NAME': 'Amit Sharma', 'DEPARTMENT': 'Engineering', 'SALARY': 85000}
{'ID': 102, 'NAME': 'Priya Singh', 'DEPARTMENT': 'Marketing', 'SALARY': 62000}
{'ID': 103, 'NAME': 'Rohit Verma', 'DEPARTMENT': 'Sales', 'SALARY': 55000}
{'ID': 104, 'NAME': 'Neha Gupta', 'DEPARTMENT': 'HR', 'SALARY': 75000}
{'ID': 105, 'NAME': 'Vikram Patel', 'DEPARTMENT': 'Engineering', 'SALARY': 90000}
{'ID': 106, 'NAME': 'Deepak Kumar', 'DEPARTMENT': 'Finance', 'SALARY': 82000}
{'ID': 107, 'NAME': 'Sneha Reddy', 'DEPARTMENT': 'Engineering', 'SALARY': 78000}
{'ID': 108, 'NAME': 'Rohan Das', 'DEPARTMENT': 'Operations', 'SALARY': 48000}
{'ID': 109, 'NAME': 'Anjali Mehra', 'DEPARTMENT': 'Marketing', 'SALARY': 65000}
{'ID': 110, 'NAME': 'Karthik Iyer', 'DEPARTMENT': 'Finance', 'SALARY': 95000}


# PART 4 - Langchain SQL Toolkit & Agent

## Task 5 : Initialize SQL Toolkit

In [35]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
db = SQLDatabase.from_uri('sqlite:///company.db')

In [37]:
groq_model = ChatGroq(model='llama-3.1-8b-instant')

In [38]:
sql_toolkit = SQLDatabaseToolkit(db = db,llm=groq_model)

In [39]:
tools = sql_toolkit.get_tools()

In [40]:
for tool in tools:
    print(tool.name," ---> ",tool.description)

sql_db_query  --->  Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.
sql_db_schema  --->  Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3
sql_db_list_tables  --->  Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker  --->  Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


## Task 6 :  Create SQL Agent

In [45]:
from langchain_classic.agents.agent_types import AgentType
from langchain_classic.agents.agent_toolkits import create_sql_agent
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.agent_toolkits import SQLDatabaseToolkit

In [53]:
sql_agent = create_sql_agent(llm=groq_model,db=db,agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,verbose=True,max_execution_time=4,early_stopping_method='force')

In [55]:
# agent_ex = AgentExecutor(agent=sql_agent,tools=sql_toolkit,verbose=True,max_iterations=4,early_stopping_method="force")

In [56]:
response = sql_agent.invoke({'input':'Give me the names of all employees'})



> Entering new SQL Agent Executor chain...
Question: Give me the names of all employees
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: EMPLOYEES, SALESQuestion: Give me the names of all employees
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: EMPLOYEES, SALESQuestion: Give me the names of all employees
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: EMPLOYEES, SALESQuestion: Give me the names of all employees
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: EMPLOYE

In [57]:
response['output']

'Agent stopped due to iteration limit or time limit.'